# P9-2 · Train ở quy mô lớn

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
model = nn.Linear(16, 4); opt = torch.optim.SGD(model.parameters(), lr=0.1); loss_fn = nn.CrossEntropyLoss()
loader = [(torch.randn(8, 16), torch.randint(0, 4, (8,))) for _ in range(8)]     # 8 micro-batch, mỗi cái 8 mẫu

accum = 4
opt.zero_grad()
for i, (x, y) in enumerate(loader):
    loss = loss_fn(model(x), y) / accum          # chia số bước tích lũy
    loss.backward()                              # gradient cộng dồn vào .grad
    if (i + 1) % accum == 0:
        opt.step(); opt.zero_grad()
        print(f"cập nhật sau micro-batch {i + 1}, batch hiệu dụng {accum * 8}")

cập nhật sau micro-batch 4, batch hiệu dụng 32
cập nhật sau micro-batch 8, batch hiệu dụng 32


In [ ]:
from torch.utils.checkpoint import checkpoint

class Net(nn.Module):
    def __init__(self, n=6, d=256):
        super().__init__()
        self.blocks = nn.ModuleList(nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d)) for _ in range(n))
    def forward(self, x):
        for block in self.blocks:
            x = checkpoint(block, x, use_reentrant=False)   # không lưu activation bên trong block
        return x
# với model transformers: model.gradient_checkpointing_enable()

net, x = Net(), torch.randn(64, 256, requires_grad=True)
net(x).sum().backward()                                    # backward tự tính lại forward từng block
print("gradient OK:", net.blocks[0][0].weight.grad.shape)

gradient OK: torch.Size([1024, 256])


## Bài tập 4: Train DDP {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
# Cần máy có 2 GPU: lưu thành train_ddp.py rồi chạy torchrun --nproc_per_node=2 train_ddp.py
import os, torch, torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler, TensorDataset

def main():
    dist.init_process_group("nccl")
    rank, local = dist.get_rank(), int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local)
    ds = TensorDataset(torch.randn(10000, 128), torch.randint(0, 10, (10000,)))
    sampler = DistributedSampler(ds, shuffle=True)
    dl = DataLoader(ds, batch_size=64, sampler=sampler)
    model = DDP(torch.nn.Sequential(torch.nn.Linear(128, 256), torch.nn.ReLU(), torch.nn.Linear(256, 10)).cuda(), device_ids=[local])
    opt = torch.optim.AdamW(model.parameters(), 1e-3)
    for epoch in range(3):
        sampler.set_epoch(epoch)                       # xáo khác nhau mỗi epoch
        for x, y in dl:
            x, y = x.cuda(), y.cuda()
            loss = torch.nn.functional.cross_entropy(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()   # all-reduce gradient tự động
        if rank == 0:
            print(epoch, loss.item())
            torch.save(model.module.state_dict(), "ckpt.pt")
    dist.destroy_process_group()

if __name__ == "__main__":
    main()
# chạy: torchrun --nproc_per_node=2 train_ddp.py

## Bài tập 5: Gradient accumulation cho cùng gradient {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 3))
X, y = torch.randn(32, 10), torch.randint(0, 3, (32,))
loss_fn = nn.CrossEntropyLoss()

def grad_vector():
    return torch.cat([p.grad.flatten() for p in model.parameters()])
# TODO: g_full (batch 32); g_acc (4 batch 8, loss / 4); g_sai (4 batch 8, không chia)

In [ ]:
model.zero_grad(); loss_fn(model(X), y).backward(); g_full = grad_vector().clone()
model.zero_grad()
for xb, yb in zip(X.split(8), y.split(8)):
    (loss_fn(model(xb), yb) / 4).backward()
g_acc = grad_vector().clone()
model.zero_grad()
for xb, yb in zip(X.split(8), y.split(8)):
    loss_fn(model(xb), yb).backward()
g_sai = grad_vector().clone()
print("cộng dồn đúng:", (g_full - g_acc).abs().max().item(), "| quên chia: gấp", round((g_sai.norm() / g_full.norm()).item(), 2), "lần")

cộng dồn đúng: 1.1175870895385742e-08 | quên chia: gấp 4.0 lần


## Bài tập 6: Gradient checkpointing đổi tính toán lấy bộ nhớ {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import torch, torch.nn as nn
from torch.utils.checkpoint import checkpoint
torch.manual_seed(0)
blocks = nn.ModuleList([nn.Sequential(nn.Linear(512, 512), nn.GELU()) for _ in range(12)])
x = torch.randn(64, 512)

def forward(x, dung_ckpt):
    for b in blocks:
        x = checkpoint(b, x, use_reentrant=False) if dung_ckpt else b(x)
    return x.pow(2).mean()

def do_bo_nho(dung_ckpt):
    tong = [0]
    def pack(t):
        tong[0] += t.numel() * t.element_size(); return t
    with torch.autograd.graph.saved_tensors_hooks(pack, lambda t: t):
        loss = forward(x, dung_ckpt)
    blocks.zero_grad(); loss.backward()
    return tong[0], torch.cat([p.grad.flatten() for p in blocks.parameters()]).clone()
# TODO: mb_thuong, g1 = do_bo_nho(False); mb_ckpt, g2 = do_bo_nho(True)

In [ ]:
mb_thuong, g1 = do_bo_nho(False)
mb_ckpt, g2 = do_bo_nho(True)
print(f"thường: {mb_thuong / 1e6:.1f} MB | checkpoint: {mb_ckpt / 1e6:.1f} MB | gradient giống: {torch.allclose(g1, g2, atol=1e-6)}")

thường: 14.8 MB | checkpoint: 1.7 MB | gradient giống: True


## Bài tập 7: Mô phỏng DDP: trung bình gradient giữa các GPU {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import copy, torch, torch.nn as nn
torch.manual_seed(0)
goc = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 2))
X, y = torch.randn(64, 8), torch.randint(0, 2, (64,))
ban = [copy.deepcopy(goc) for _ in range(4)]
opts = [torch.optim.SGD(m.parameters(), lr=0.1) for m in ban]
mot_minh = copy.deepcopy(goc); opt1 = torch.optim.SGD(mot_minh.parameters(), lr=0.1)
# TODO: mỗi bản backward trên phần của nó; all-reduce (trung bình) gradient; step; so tham số

In [ ]:
for m, xb, yb in zip(ban, X.split(16), y.split(16)):
    m.zero_grad(); nn.functional.cross_entropy(m(xb), yb).backward()
for params in zip(*[list(m.parameters()) for m in ban]):
    tb = torch.stack([p.grad for p in params]).mean(0)
    for p in params:
        p.grad = tb.clone()
for o in opts:
    o.step()
opt1.zero_grad(); nn.functional.cross_entropy(mot_minh(X), y).backward(); opt1.step()

vec = lambda m: torch.cat([p.detach().flatten() for p in m.parameters()])
giong_nhau = all(torch.equal(vec(ban[0]), vec(m)) for m in ban[1:])
giong_mot_minh = torch.allclose(vec(ban[0]), vec(mot_minh), atol=1e-6)
print("4 bản giống nhau:", giong_nhau, "| giống train một mình:", giong_mot_minh)

4 bản giống nhau: True | giống train một mình: True


## Bài tập 8: Bộ nhớ mỗi GPU theo từng mức ZeRO {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
def bo_nho_gpu(n_ty, so_gpu, stage):
    # TODO: (trọng số, gradient, optimizer) mỗi phần chia cho so_gpu nếu stage đủ cao
    return 0.0

In [ ]:
def bo_nho_gpu(n_ty, so_gpu, stage):
    w = 2 * n_ty / (so_gpu if stage >= 3 else 1)
    g = 2 * n_ty / (so_gpu if stage >= 2 else 1)
    o = 12 * n_ty / (so_gpu if stage >= 1 else 1)
    return w + g + o

for s in range(4):
    print(f"stage {s}: {bo_nho_gpu(7, 8, s):6.1f} GB mỗi GPU")

stage 0:  112.0 GB mỗi GPU
stage 1:   38.5 GB mỗi GPU
stage 2:   26.2 GB mỗi GPU
stage 3:   14.0 GB mỗi GPU
